In [7]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 0. Array รายชื่อ 77 จังหวัดของประเทศไทย (รูปแบบตัวพิมพ์ใหญ่ภาษาอังกฤษ)
# ---------------------------------------------------------
provinces_thailand = [
    "BANGKOK", "AMNAT CHAROEN", "ANG THONG", "BUENG KAN", "BURI RAM", "CHACHOENGSAO",
    "CHAI NAT", "CHAIYAPHUM", "CHANTHABURI", "CHIANG MAI", "CHIANG RAI", "CHON BURI",
    "CHUMPHON", "KALASIN", "KAMPHAENG PHET", "KANCHANABURI", "KHON KAEN", "KRABI",
    "LAMPANG", "LAMPHUN", "LOEI", "LOP BURI", "MAE HONG SON", "MAHA SARAKHAM",
    "MUKDAHAN", "NAKHON NAYOK", "NAKHON PATHOM", "NAKHON PHANOM", "NAKHON RATCHASIMA",
    "NAKHON SAWAN", "NAKHON SI THAMMARAT", "NAN", "NARATHIWAT", "NONG BUA LAMPHU",
    "NONG KHAI", "NONTHABURI", "PATHUM THANI", "PATTANI", "PHANG NGA", "PHATTHALUNG",
    "PHAYAO", "PHETCHABUN", "PHETCHABURI", "PHICHIT", "PHITSANULOK", "PHRA NAKHON SI AYUTTHAYA",
    "PHRAE", "PHUKET", "PRACHIN BURI", "PRACHUAP KHIRI KHAN", "RANONG", "RATCHABURI",
    "RAYONG", "ROI ET", "SA KAEO", "SAKON NAKHON", "SAMUT PRAKAN", "SAMUT SAKHON",
    "SAMUT SONGKHRAM", "SARABURI", "SATUN", "SING BURI", "SI SA KET", "SONGKHLA",
    "SUKHOTHAI", "SUPHAN BURI", "SURAT THANI", "SURIN", "TAK", "TRANG",
    "TRAT", "UBON RATCHATHANI", "UDON THANI", "UTHAI THANI", "UTTARADIT", "YALA", "YASOTHON"
]

# ---------------------------------------------------------
# Step 1: แปลง XML Climate (ข้อมูลอุตุนิยมวิทยา TMD)
# ---------------------------------------------------------
# สร้าง Set รายชื่อจังหวัดแบบตัดช่องว่าง เพื่อเปรียบเทียบได้ยืดหยุ่นขึ้น (กันปัญหาเว้นวรรคไม่เท่ากัน)
provinces_set = {p.replace(" ", "").upper() for p in provinces_thailand}

def parse_climate_xml(xml_file_path):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()
    
    months = ['January', 'February', 'March', 'April', 'May', 'June', 
              'July', 'August', 'September', 'October', 'November', 'December']
    
    station_data = {}
    
    for station in root.findall('StationClimateNormal'):
        st_name = station.find('StationName').text.strip()
        
        # -------------------------------------------------------------
        # [เพิ่มจุดนี้] ตรวจสอบว่าชื่อสถานีคือจังหวัดหรือไม่ ถ้าไม่ใช่ (เป็นอำเภอ) ให้ข้าม
        # -------------------------------------------------------------
        st_name_clean = st_name.replace(" ", "").upper()
        if st_name_clean not in provinces_set:
            continue  # ข้ามสถานีที่เป็นอำเภอหรือไม่ใช่อยู่ใน 77 จังหวัด
        
        if st_name not in station_data:
            station_data[st_name] = {'location': st_name}
        
        for nv in station.findall('NormalValue'):
            elem_elem = nv.find('Element')
            if elem_elem is None or elem_elem.text is None:
                continue
            
            elem_name = elem_elem.text.strip()
            vals = []
            for m in months:
                val_tag = nv.find(f'ValueOn{m}')
                if val_tag is not None and val_tag.text:
                    vals.append(float(val_tag.text))
            
            if not vals:
                continue
                
            if elem_name == 'Mean Dry-bulb Temperature':
                station_data[st_name]['temperature'] = np.mean(vals)
            elif elem_name == 'Mean Relative Humidity':
                station_data[st_name]['humidity'] = np.mean(vals)
            elif elem_name == 'Total Rainfall':
                station_data[st_name]['rainfall'] = np.sum(vals)
                
    return pd.DataFrame(list(station_data.values()))

# ---------------------------------------------------------
# Step 2: แปลง XML Soil (ข้อมูลดิน)
# ---------------------------------------------------------
def parse_soil_xml(xml_file_path):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()
    
    soil_records = []
    for soil in root.findall('SoilRecord'):
        location = soil.find('location').text.strip()
        ph = float(soil.find('ph').text)
        organic_matter = float(soil.find('organic_matter').text)
        soil_type = soil.find('soil_type').text.strip()
        
        soil_records.append({
            'location': location,
            'ph': ph,
            'organic_matter': organic_matter,
            'soil_type': soil_type
        })
        
    return pd.DataFrame(soil_records)

# ---------------------------------------------------------
# Step 3: ผสานข้อมูล XML กับ CSV เดิม
# ---------------------------------------------------------
# 1. ดึงข้อมูลจากไฟล์ XML
df_climate = parse_climate_xml('climate_normal.xml')

# 2. โหลด CSV เดิม
df_original = pd.read_csv('Crop_recommendation.csv') 

# ตรวจสอบคอลัมน์ location
if 'location' not in df_original.columns:
    locations_list = df_climate['location'].unique()
    np.random.seed(42)
    df_original['location'] = np.random.choice(locations_list, size=len(df_original))

# 3. Merge ข้อมูล Climate เข้าด้วยกัน
df_merged = pd.merge(df_original, df_climate, on='location', how='left', suffixes=('_original', '_xml'))

# -------------------------------------------------------------
# [เพิ่มจุดนี้] กรองลบ Row ที่ไม่ใช่จังหวัดออก
# -------------------------------------------------------------
provinces_set = {p.replace(" ", "").upper() for p in provinces_thailand}

# สร้างคอลัมน์ชั่วคราวเพื่อเช็คชื่อจังหวัด
df_merged['temp_check'] = df_merged['location'].astype(str).str.replace(" ", "").str.upper()

# เอาเฉพาะแถวที่อยู่ในรายชื่อจังหวัด
df_merged = df_merged[df_merged['temp_check'].isin(provinces_set)].copy()

# ลบคอลัมน์ชั่วคราวออก
df_merged = df_merged.drop(columns=['temp_check'])

# เติมค่าว่างกรณีมีข้อมูลไม่สมบูรณ์
df_merged = df_merged.fillna(df_merged.mean(numeric_only=True))

# 4. บันทึกเป็น CSV ใหม่
df_merged.to_csv('enriched_crop_dataset.csv', index=False)
print(f"บันทึกไฟล์เรียบร้อย! จำนวนข้อมูลที่เหลือ: {len(df_merged)} แถว")



บันทึกไฟล์เรียบร้อย! จำนวนข้อมูลที่เหลือ: 2200 แถว
